# Asteroid Risk Prediction — ZOSA Data Science
Train a classifier to predict potentially hazardous asteroids.

Uses the Gold asteroid risk table, engineers ML features,
trains three models with MLflow tracking, and saves predictions.

## 1 — Load Data & Feature Engineering

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

# Load gold table
df_spark = spark.table("gold_asteroid_risk")
df = df_spark.toPandas()

# Feature engineering
df["size_speed_ratio"] = df["diameter_km"] / (df["velocity_km_s"] + 0.001)
df["proximity_score"] = 1.0 / (df["miss_distance_au"] + 0.001)
df["kinetic_energy_proxy"] = df["diameter_km"] ** 2 * df["velocity_km_s"]

feature_cols = ["diameter_km", "velocity_km_s", "miss_distance_au",
                "hazard_score", "size_speed_ratio", "proximity_score",
                "kinetic_energy_proxy"]

X = df[feature_cols].fillna(0)
y = df["is_hazardous"].astype(int)

print(f"Dataset: {len(df)} samples, {len(feature_cols)} features")
print(f"Class balance: {y.value_counts().to_dict()}")

## 2 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 3 — Train Three Models with MLflow

In [ ]:
mlflow.set_experiment("asteroid-risk-prediction")

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
}

results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        auc = roc_auc_score(y_test, proba)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", auc)
        mlflow.sklearn.log_model(model, artifact_path="model")

        results[name] = {"accuracy": acc, "f1": f1, "auc": auc, "model": model}
        print(f"{name:25s} — Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

## 4 — Compare & Select Best Model

In [ ]:
best_name = max(results, key=lambda k: results[k]["auc"])
best = results[best_name]
print(f"\n🏆 Best model: {best_name}")
print(f"   AUC: {best['auc']:.4f} | F1: {best['f1']:.4f} | Accuracy: {best['accuracy']:.4f}")

best_model = best["model"]

## 5 — Save Predictions to Gold Layer

In [ ]:
# Score the full dataset with the best model
df["predicted_hazardous"] = best_model.predict(X)
df["hazard_probability"] = best_model.predict_proba(X)[:, 1]

pred_spark = spark.createDataFrame(df)
pred_spark.write.mode("overwrite").format("delta").saveAsTable("gold_asteroid_predictions")
print(f"✅ gold_asteroid_predictions: {pred_spark.count()} rows saved")

## 6 — Feature Importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
else:
    importances = np.zeros(len(feature_cols))

feat_df = pd.DataFrame({"feature": feature_cols, "importance": importances}) \
            .sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(feat_df["feature"], feat_df["importance"], color="#0078D4")
ax.set_xlabel("Importance")
ax.set_title(f"Feature Importance — {best_name}")
plt.tight_layout()
plt.show()

print("\n✅ Asteroid risk model pipeline complete.")